[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/00_p10_training_loop.ipynb)

# 🔵 Prerequisite 10: Training Loop & Randomness

Most "implement X" problems end with "…and train it". Several more (Dropout, top-k sampling, speculative decoding) are about **randomness**: drawing masks or tokens with the right probabilities, **reproducibly**.

This notebook covers the canonical training loop, how optimizers expose their learning rate, and how to use PyTorch's random number generators so that tests (and experiments) give the same result every run.

### What you'll learn
1. The canonical loop: `zero_grad → forward → loss → backward → step`
2. Optimizer `param_groups` and changing the learning rate
3. Seeds and `torch.Generator` for reproducible randomness
4. Random draws: `rand`, `bernoulli`, `multinomial`, `randperm`
5. Shuffling and mini-batching a dataset
6. Evaluation: `model.eval()` plus `torch.no_grad()`

### Exercises
| # | Function | Shows up in |
|---|----------|-------------|
| 1 | `train_steps(model, optimizer, loss_fn, x, y, steps)` | #40 Linear Regression, #31 Gradient Accumulation |
| 2 | `set_lr_(optimizer, lr)` | #30 Cosine LR Scheduler |
| 3 | `keep_mask(shape, p_drop, generator)` | #17 Dropout |
| 4 | `sample_tokens(probs, n, generator)` | #32 Top-k/p Sampling, #34 Speculative Decoding |
| 5 | `shuffled_batches(X, y, batch_size, generator)` | mini-batch training, #31 |

### Rules
- **No Python loops** over tensor elements. Looping over training steps or batches is fine.
- Use the `generator` you are given for every random draw.

### How to use this notebook
1. **Run the lesson cells** (sections 1, 2, …) from top to bottom. They are demos: change them and re-run.
2. **Do the exercises in order.** Each one has its own `✏️ YOUR IMPLEMENTATION HERE` cell, followed by a debug cell for checking shapes before you submit.
3. **Submit at any time** with the `check("p10_training_loop")` cell at the bottom. It grades every exercise at once, and each test is named after its exercise, so a partial score shows which ones are left.
4. **Stuck?** `hint("p10_training_loop")` gives one hint per exercise; `00_p10_training_loop_solution.ipynb` has the reference code.
5. **Already comfortable with this skill?** Prerequisites are optional. Skip straight to the problems listed in **🔓 What this unlocks** at the bottom.

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass

In [ ]:
import torch
from torch import nn

## 1. The canonical training loop

```python
for step in range(steps):
    optimizer.zero_grad()        # 1. clear old gradients (they accumulate, see P7)
    pred = model(x)              # 2. forward
    loss = loss_fn(pred, y)      # 3. loss (a scalar)
    loss.backward()              # 4. gradients into p.grad
    optimizer.step()             # 5. update the parameters
```
Log the loss with `loss.item()`. Keeping the tensor `loss` in a list keeps its whole graph alive.

In [ ]:
torch.manual_seed(0)
model = nn.Linear(3, 1)
opt = torch.optim.SGD(model.parameters(), lr=0.1)
x = torch.randn(32, 3)
y = x @ torch.tensor([[1.0], [-2.0], [0.5]]) + 0.3
for step in range(50):
    opt.zero_grad()
    loss = nn.functional.mse_loss(model(x), y)
    loss.backward()
    opt.step()
print(f"final loss {loss.item():.5f}", model.weight.data, model.bias.data)

## 2. `param_groups` and the learning rate

An optimizer keeps its hyperparameters in `optimizer.param_groups`, a list of dicts (one per group of parameters). LR schedulers (#30) just write a new `"lr"` into every group before each step.

In [ ]:
opt = torch.optim.SGD([
    {"params": [torch.zeros(2, requires_grad=True)], "lr": 0.1},
    {"params": [torch.zeros(3, requires_grad=True)], "lr": 0.01},
])
print([g["lr"] for g in opt.param_groups])
for g in opt.param_groups:
    g["lr"] = 0.5
print([g["lr"] for g in opt.param_groups])

## 3. Seeds and generators

`torch.manual_seed(s)` resets the **global** generator, which every random call uses by default.
A `torch.Generator` is a **private** random stream. Pass it as `generator=g`, and your function's randomness no longer depends on what other code drew before it. Tests use this to check that your output is reproducible.

In [ ]:
g1 = torch.Generator().manual_seed(42)
g2 = torch.Generator().manual_seed(42)
torch.manual_seed(0); a = torch.rand(3, generator=g1)
torch.manual_seed(999); b = torch.rand(3, generator=g2)   # global seed doesn't matter
print(a, b, torch.equal(a, b))

## 4. Random draws

| Want | Use |
|------|-----|
| Uniform in `[0, 1)` | `torch.rand(shape, generator=g)` |
| Coin flips with probability `p` | `torch.rand(shape, generator=g) < p`, or `torch.bernoulli(probs, generator=g)` |
| Categorical samples | `torch.multinomial(probs, n, replacement=True, generator=g)` |
| A random permutation | `torch.randperm(n, generator=g)` |

`multinomial` takes **probabilities**, or any non-negative weights. They don't need to sum to 1, but they must not be logits.

In [ ]:
g = torch.Generator().manual_seed(0)
probs = torch.tensor([0.7, 0.2, 0.1])
draws = torch.multinomial(probs, 10_000, replacement=True, generator=g)
print(torch.bincount(draws, minlength=3) / 10_000)    # close to [0.7, 0.2, 0.1]
print(torch.randperm(8, generator=g))

## 5. Shuffling and mini-batches

Shuffle once per epoch with a permutation, then slice it into batches. Index `X` and `y` with the **same** permutation so the pairs stay together. `torch.split` slices along dim 0.

In [ ]:
X = torch.arange(10).float()[:, None]
g = torch.Generator().manual_seed(1)
perm = torch.randperm(10, generator=g)
print([chunk.flatten().tolist() for chunk in torch.split(X[perm], 4)])   # sizes 4, 4, 2

## 6. Evaluation

For evaluation, switch the model's behavior with `model.eval()` (Dropout off, BatchNorm uses running stats; see P8), **and** stop building graphs with `torch.no_grad()`. These are two separate things, and you usually want both.

In [ ]:
model = nn.Sequential(nn.Linear(4, 4), nn.Dropout(0.5))
model.eval()
with torch.no_grad():
    out = model(torch.randn(2, 4))
print(out.requires_grad, model.training)   # False False
model.train()

---
# ✏️ Exercises

Implement the five functions below. Each one has a debug cell you can use before you submit.

### Exercise 1: `train_steps(model, optimizer, loss_fn, x, y, steps)`

Run the canonical training loop for `steps` full-batch steps.

- `model`: an `nn.Module`; `optimizer`: built on `model.parameters()`; `loss_fn(pred, y)` returns a scalar
- each step: clear gradients, forward, loss, backward, optimizer step
- returns: a list of `steps` **Python floats**, the loss at each step (computed before that step's update)

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

def train_steps(model: nn.Module, optimizer, loss_fn, x: torch.Tensor, y: torch.Tensor,
                steps: int) -> list[float]:
    pass  # Replace this

In [ ]:
torch.manual_seed(0)
model = nn.Linear(3, 1)
x = torch.randn(32, 3); y = x.sum(dim=1, keepdim=True)
losses = train_steps(model, torch.optim.SGD(model.parameters(), lr=0.1), nn.MSELoss(), x, y, steps=30)
print(f"{losses[0]:.3f} -> {losses[-1]:.5f}", type(losses[0]))   # expect a falling loss, <class 'float'>

### Exercise 2: `set_lr_(optimizer, lr)`

Set the learning rate of **every** parameter group of an optimizer, in place.

- returns: nothing

*An LR scheduler (#30) computes `lr` for the current step and does exactly this.*

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

def set_lr_(optimizer, lr: float) -> None:
    pass  # Replace this

In [ ]:
opt = torch.optim.SGD([{"params": [torch.zeros(1, requires_grad=True)], "lr": 0.1},
                       {"params": [torch.zeros(1, requires_grad=True)], "lr": 0.2}])
set_lr_(opt, 0.05)
print([g["lr"] for g in opt.param_groups])   # expect [0.05, 0.05]

### Exercise 3: `keep_mask(shape, p_drop, generator)`

Draw a dropout **keep** mask: each entry is `True` with probability `1 - p_drop`, independently.

- `shape`: a tuple of ints
- `p_drop`: float in `[0, 1]`; `p_drop=0` keeps everything, `p_drop=1` keeps nothing
- `generator`: a `torch.Generator`; use it for the random draw
- returns: a `bool` tensor of the given shape

*#17 Dropout multiplies by this mask and rescales by `1 / (1 - p)`.*

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

def keep_mask(shape: tuple, p_drop: float, generator: torch.Generator) -> torch.Tensor:
    pass  # Replace this

In [ ]:
g = torch.Generator().manual_seed(0)
m = keep_mask((1000, 100), 0.3, g)
print(m.dtype, m.float().mean())   # expect torch.bool, about 0.7

### Exercise 4: `sample_tokens(probs, n, generator)`

Draw `n` token ids, with replacement, from a categorical distribution.

- `probs`: `(V,)` non-negative probabilities summing to 1
- `n`: number of samples
- `generator`: a `torch.Generator`; use it for the draw
- returns: `(n,)` long tensor of token ids in `[0, V)`

*#32 samples the next token from the filtered distribution; #34 re-samples from the adjusted one.*

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

def sample_tokens(probs: torch.Tensor, n: int, generator: torch.Generator) -> torch.Tensor:
    pass  # Replace this

In [ ]:
g = torch.Generator().manual_seed(0)
draws = sample_tokens(torch.tensor([0.5, 0.0, 0.3, 0.2]), 20_000, g)
print(torch.bincount(draws, minlength=4) / 20_000)   # expect about [0.5, 0.0, 0.3, 0.2]

### Exercise 5: `shuffled_batches(X, y, batch_size, generator)`

Shuffle a dataset once and split it into mini-batches.

- `X`: `(N, ...)`, `y`: `(N, ...)`, paired row by row
- `batch_size`: int; the last batch may be smaller
- `generator`: a `torch.Generator`; use it for the shuffle
- returns: a list of `(X_batch, y_batch)` tuples that together contain **every row exactly once**, in a random order, with `X[i]` still paired with `y[i]`

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

def shuffled_batches(X: torch.Tensor, y: torch.Tensor, batch_size: int,
                     generator: torch.Generator) -> list[tuple[torch.Tensor, torch.Tensor]]:
    pass  # Replace this

In [ ]:
X = torch.arange(10).float()[:, None]
y = X.squeeze(1) * 10
g = torch.Generator().manual_seed(0)
for xb, yb in shuffled_batches(X, y, 4, g):
    print(xb.flatten().tolist(), yb.tolist())   # sizes 4, 4, 2; y is always 10 * x

<!-- Generated from scripts/prereqs/content/p10_training_loop.py by `make prereqs`. Edit that file instead of this notebook; manual edits are overwritten. -->
## ✅ Submit

Run the judge on **all** exercises in this notebook. Each test is labelled with the exercise it checks.

In [ ]:
from torch_judge import check, hint
check("p10_training_loop")

In [ ]:
# 💡 Stuck? Uncomment for a nudge
# hint("p10_training_loop")

---
## 🔓 What this unlocks

With **Training Loop & Randomness** in your toolkit, you're ready for these problems:

| # | Problem | Difficulty | What you'll reuse | Links |
|:---:|---|:---:|---|---|
| #17 | Dropout | 🟢 Easy | `keep_mask`: the random keep/drop decision behind Dropout | [Open](17_dropout.ipynb) · <a href="https://github.com/duoan/TorchCode/blob/master/templates/17_dropout.ipynb" target="_blank">GitHub</a> · <a href="https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/17_dropout.ipynb" target="_blank">Colab</a> |
| #29 | Adam Optimizer | 🟡 Medium | The `zero_grad` / `step` protocol your optimizer must follow | [Open](29_adam.ipynb) · <a href="https://github.com/duoan/TorchCode/blob/master/templates/29_adam.ipynb" target="_blank">GitHub</a> · <a href="https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/29_adam.ipynb" target="_blank">Colab</a> |
| #30 | Cosine LR Scheduler | 🟡 Medium | `set_lr_`: a scheduler writes the new LR into every param group | [Open](30_cosine_lr.ipynb) · <a href="https://github.com/duoan/TorchCode/blob/master/templates/30_cosine_lr.ipynb" target="_blank">GitHub</a> · <a href="https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/30_cosine_lr.ipynb" target="_blank">Colab</a> |
| #31 | Gradient Accumulation | 🟢 Easy | The training loop, with micro-batches like `shuffled_batches` | [Open](31_gradient_accumulation.ipynb) · <a href="https://github.com/duoan/TorchCode/blob/master/templates/31_gradient_accumulation.ipynb" target="_blank">GitHub</a> · <a href="https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/31_gradient_accumulation.ipynb" target="_blank">Colab</a> |
| #32 | Top-k / Top-p Sampling | 🟡 Medium | `sample_tokens`: drawing the next token with `multinomial` | [Open](32_topk_sampling.ipynb) · <a href="https://github.com/duoan/TorchCode/blob/master/templates/32_topk_sampling.ipynb" target="_blank">GitHub</a> · <a href="https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/32_topk_sampling.ipynb" target="_blank">Colab</a> |
| #34 | Speculative Decoding | 🔴 Hard | Random acceptance with `torch.rand`, and re-sampling with `multinomial` | [Open](34_speculative_decoding.ipynb) · <a href="https://github.com/duoan/TorchCode/blob/master/templates/34_speculative_decoding.ipynb" target="_blank">GitHub</a> · <a href="https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/34_speculative_decoding.ipynb" target="_blank">Colab</a> |
| #40 | Linear Regression | 🟡 Medium | `train_steps`: the `nn.Linear` + SGD method | [Open](40_linear_regression.ipynb) · <a href="https://github.com/duoan/TorchCode/blob/master/templates/40_linear_regression.ipynb" target="_blank">GitHub</a> · <a href="https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/40_linear_regression.ipynb" target="_blank">Colab</a> |

🗺️ See the full recommended order in <a href="https://github.com/duoan/TorchCode/blob/master/LEARNING_PATH.md" target="_blank">LEARNING_PATH.md</a>.